## 1. Setup Snowflake connection and load data

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

### Load Libraries

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run "./00_cohort_config"

### Get Data from Scope

[REDACTED]

[REDACTED]

[REDACTED]

[REDACTED]



# Cohort Configuration

Shared config notebook for the 6 patient journey validation notebooks. 
Defines the offset, data boundaries, cohort labels (C1–C22), and the cohort assignment function.

**Usage:** Each patient journey notebook should `%run` this notebook early on.

### Define Snowflake Connection Function
Choose appropiate below items based on the Brand
- Role
- Warehouse
- Schema
- Database

OFFSET_DAY      = 0
DATA_START_DATE = 2023-01-01
COHORT_ORIGIN   = 2025-01-01
COHORT_LAST     = 2026-10-01


In [0]:

# VALIDATION: read from validation outputs (not production)
# NOTE: lab data now reads notebook 01's PROCESSED output (forward-filled, cohort-month grain)
# The original read the raw input (kerendia_nba_3_0_lab_query_output) which was at daily grain
df_lab_pat_journey = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_lab_journey_validation")

df_dx_pat_journey = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_dx_journey_validation")

df_rx_pat_journey = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_rx_journey_validation")

df_t2d_patient_pat_journey = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_t2d_patient_validation")

df_px_pat_journey = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_px_validation")

cohort_label,cohort_ref_month,cohort_start,cohort_end
C1,2025-01-01,2025-01-01,2025-02-01
C2,2025-02-01,2025-02-01,2025-03-01
C3,2025-03-01,2025-03-01,2025-04-01
C4,2025-04-01,2025-04-01,2025-05-01
C5,2025-05-01,2025-05-01,2025-06-01
C6,2025-06-01,2025-06-01,2025-07-01
C7,2025-07-01,2025-07-01,2025-08-01
C8,2025-08-01,2025-08-01,2025-09-01
C9,2025-09-01,2025-09-01,2025-10-01
C10,2025-10-01,2025-10-01,2025-11-01



22 cohorts defined (C1 – C22)


Testing with current OFFSET_DAY = 0
-------------------------------------------------------
  2025-01-01 → 2025-01-01  (expected 2025-01-01)  [PASS]

Config notebook ready. All 6 patient journey notebooks can %run this.


In [0]:
print('df_lab_pat_journey',df_lab_pat_journey.count(), len(df_lab_pat_journey.columns))
print('df_dx_pat_journey',df_dx_pat_journey.count(), len(df_dx_pat_journey.columns))
print('df_rx_pat_journey',df_rx_pat_journey.count(), len(df_rx_pat_journey.columns))
print('df_t2d_patient_pat_journey',df_t2d_patient_pat_journey.count(), len(df_t2d_patient_pat_journey.columns))
print('df_px_pat_journey',df_px_pat_journey.count(), len(df_px_pat_journey.columns))


df_lab_pat_journey 6911 5
df_dx_pat_journey 36124 5
df_rx_pat_journey 113587 5
df_t2d_patient_pat_journey 76999091 2
df_px_pat_journey 1481244 3


## 2. Combine Patinet Journey Table

In [0]:
df_spine_pat_journey = (
  df_lab_pat_journey
  .join(df_dx_pat_journey, how = "full", on = ["PATIENT_ID", "DATE"])
  .join(df_rx_pat_journey, how = "full", on = ["PATIENT_ID", "DATE"])
  .orderBy("PATIENT_ID", "DATE")
  .withColumn("DX_STAGE_MAX", col("DX_STAGE_UPDATED"))

  .withColumn("GFR_DX_STAGE", F.coalesce(col("DX_STAGE_MAX"), col("GFR_STAGE")))
 
  .withColumn("GFR_UACR_DX_STAGE", F.coalesce(col("DX_STAGE_MAX"), col("GFR_STAGE"), col("UACR_STAGE")))
)

df_spine_pat_journey.display()

PATIENT_ID,DATE,GFR_STAGE,UACR_STAGE,cohort_label,DX_STAGE,DX_STAGE_UPDATED,cohort_label,TREATMENT_CATEGORY,CATEGORY,cohort_label,DX_STAGE_MAX,GFR_DX_STAGE,GFR_UACR_DX_STAGE
1000151945,2023-01-01,null,null,null,null,null,null,List(Combo drug),List(Combo drug),null,null,null,null
1000151945,2023-02-01,null,null,null,null,null,null,"List(Diuretic_NON_MRA, Combo drug, Insulin)","List(Insulin, Combo drug, Diuretic_NON_MRA)",null,null,null,null
1000151945,2023-03-01,null,null,null,null,null,null,"List(Diuretic_NON_MRA, Insulin, metformin , Combo drug)","List(Insulin, Diuretic_NON_MRA, Combo drug, metformin )",null,null,null,null
1000151945,2023-04-01,null,null,null,null,null,null,"List(Combo drug, Insulin, Diuretic_NON_MRA, metformin )","List(Combo drug, Insulin, metformin , Diuretic_NON_MRA)",null,null,null,null
1000151945,2023-05-01,null,null,null,null,null,null,"List(Diuretic_NON_MRA, Combo drug, metformin , Insulin)","List(Diuretic_NON_MRA, metformin , Insulin, Combo drug)",null,null,null,null
1000151945,2023-06-01,null,null,null,null,null,null,"List(Combo drug, Diuretic_NON_MRA, Insulin, metformin )","List(Combo drug, Diuretic_NON_MRA, metformin , Insulin)",null,null,null,null
1000151945,2023-07-01,null,null,null,null,null,null,"List(Diuretic_NON_MRA, metformin , Insulin, Combo drug)","List(Insulin, Combo drug, metformin , Diuretic_NON_MRA)",null,null,null,null
1000151945,2023-08-01,null,null,null,null,null,null,"List(Diuretic_NON_MRA, metformin , Insulin, Calcium channel / Beta blocker, Combo drug)","List(Insulin, Diuretic_NON_MRA, metformin , Calcium channel / Beta blocker, Combo drug)",null,null,null,null
1000151945,2023-09-01,null,null,null,null,null,null,"List(Insulin, Calcium channel / Beta blocker, Combo drug, metformin , Diuretic_NON_MRA)","List(Insulin, Combo drug, Diuretic_NON_MRA, metformin , Calcium channel / Beta blocker)",null,null,null,null
1000151945,2023-10-01,null,null,null,null,null,null,"List(Diuretic_NON_MRA, Combo drug, Insulin, metformin , Calcium channel / Beta blocker)","List(Diuretic_NON_MRA, Calcium channel / Beta blocker, Combo drug, metformin , Insulin)",null,null,null,null


In [0]:
df_spine_pat_journey_with_flag1 = (
  df_spine_pat_journey
  .join(df_t2d_patient_pat_journey, how = "left", on = "PATIENT_ID")
  .na.fill(value=0, subset=["DIABETIC_FLAG"])
)

In [0]:
df_spine_pat_journey_with_flag2 = (
  df_spine_pat_journey_with_flag1
  .join(df_px_pat_journey, how = "left", on = "PATIENT_ID")
  # Compare at month grain: truncate FIRST_DIALYSIS_DATE to month so it aligns with DATE
  # (DATE is now 1st-of-month cohort reference; without this, a mid-month dialysis would lag by 1 month)
  .withColumn("DIALYSIS_FLAG", 
              F.when(col("FIRST_DIALYSIS_DATE") < '2017-01-01', 1).otherwise(
              F.when(F.trunc(F.to_date(col("FIRST_DIALYSIS_DATE")), "month") <= col("DATE"), 1).otherwise(0)))
  .na.fill(value=0, subset=["DIALYSIS_FLAG"])
  .drop("FIRST_DIALYSIS_DATE")
)

In [0]:
df_spine_pat_journey_with_flag2.filter(col("PATIENT_ID") == 29370266).display()

PATIENT_ID,DATE,GFR_STAGE,UACR_STAGE,cohort_label,DX_STAGE,DX_STAGE_UPDATED,cohort_label,TREATMENT_CATEGORY,CATEGORY,cohort_label,DX_STAGE_MAX,GFR_DX_STAGE,GFR_UACR_DX_STAGE,DIABETIC_FLAG,DIALYSIS_FLAG
29370266,2025-09-01,null,null,null,null,null,null,"List(ACE/ ARB, Combo drug, Calcium channel / Beta blocker, MRA)","List(ACE/ ARB, Combo drug, MRA, Calcium channel / Beta blocker)",C9,null,null,null,1,0
29370266,2025-05-01,null,null,null,null,null,null,"List(Combo drug, MRA, ACE/ ARB, Diuretic_NON_MRA, Calcium channel / Beta blocker)","List(Diuretic_NON_MRA, Calcium channel / Beta blocker, MRA, ACE/ ARB, Combo drug)",C5,null,null,null,1,0
29370266,2023-02-01,null,null,null,null,null,null,"List(metformin , ACE/ ARB, Combo drug, Calcium channel / Beta blocker, MRA)","List(metformin , MRA, ACE/ ARB, Calcium channel / Beta blocker, Combo drug)",null,null,null,null,1,0
29370266,2023-11-01,null,null,null,null,null,null,"List(metformin , Combo drug, ACE/ ARB, Calcium channel / Beta blocker, MRA, SGLT2)","List(ACE/ ARB, Combo drug, metformin , Calcium channel / Beta blocker, MRA, JARDIANCE)",null,null,null,null,1,0
29370266,2024-09-01,null,null,null,null,null,null,"List(Diuretic_NON_MRA, metformin , MRA, Calcium channel / Beta blocker, Combo drug, ACE/ ARB)","List(metformin , ACE/ ARB, MRA, Calcium channel / Beta blocker, Diuretic_NON_MRA, Combo drug)",null,null,null,null,1,0
29370266,2025-07-01,null,null,null,null,null,null,"List(Diuretic_NON_MRA, MRA, Calcium channel / Beta blocker, ACE/ ARB, Combo drug)","List(Diuretic_NON_MRA, ACE/ ARB, Combo drug, MRA, Calcium channel / Beta blocker)",C7,null,null,null,1,0
29370266,2023-04-01,null,null,null,null,null,null,"List(ACE/ ARB, metformin , Combo drug, Calcium channel / Beta blocker, MRA)","List(ACE/ ARB, MRA, Combo drug, metformin , Calcium channel / Beta blocker)",null,null,null,null,1,0
29370266,2024-11-01,null,null,null,null,null,null,"List(MRA, Calcium channel / Beta blocker, Diuretic_NON_MRA, metformin , ACE/ ARB, Combo drug)","List(Calcium channel / Beta blocker, Diuretic_NON_MRA, Combo drug, MRA, ACE/ ARB, metformin )",null,null,null,null,1,0
29370266,2024-02-01,null,null,null,null,null,null,"List(Combo drug, MRA, ACE/ ARB, Calcium channel / Beta blocker, Diuretic_NON_MRA, metformin )","List(Combo drug, ACE/ ARB, Diuretic_NON_MRA, MRA, metformin , Calcium channel / Beta blocker)",null,null,null,null,1,0
29370266,2025-11-01,null,null,null,null,null,null,"List(Calcium channel / Beta blocker, ACE/ ARB, Combo drug, MRA)","List(Combo drug, MRA, Calcium channel / Beta blocker, ACE/ ARB)",C11,null,null,null,1,0



#### NPI_NUMBER, MONTH Level Patient Journey

In [0]:
df_spine_pat_journey_with_flag2.select("TREATMENT_CATEGORY").limit(5).show(truncate=False)

+------------------------------------------------------+
|TREATMENT_CATEGORY                                    |
+------------------------------------------------------+
|[Combo drug, Calcium channel / Beta blocker]          |
|[ACE/ ARB, Calcium channel / Beta blocker]            |
|[ACE/ ARB, Calcium channel / Beta blocker, metformin ]|
|[Combo drug, Diuretic_NON_MRA, SGLT2]                 |
|[Combo drug, Calcium channel / Beta blocker]          |
+------------------------------------------------------+



In [0]:
pat_new_and_total_treatments =  (
  df_spine_pat_journey_with_flag2
  # DATE is already at cohort-month grain from upstream notebooks — no need to date_trunc
  .withColumn("SVC_DT_M", col("DATE"))
  .withColumn("TOTAL_TREATMENTS", F.greatest(F.size(col("TREATMENT_CATEGORY")), F.lit(0)))
  .withColumn("TOTAL_TREATMENTS_LAST", F.lag("TOTAL_TREATMENTS", 1).over(Window.partitionBy("PATIENT_ID").orderBy("DATE")))
  .withColumn("TOTAL_NEW_TREATMENTS", F.greatest(col("TOTAL_TREATMENTS") - col("TOTAL_TREATMENTS_LAST"), F.lit(0)))
  .groupBy("PATIENT_ID", "SVC_DT_M")
  .agg(F.max("TOTAL_TREATMENTS").alias("TOTAL_TREATMENTS"),
       F.sum("TOTAL_NEW_TREATMENTS").alias("TOTAL_NEW_TREATMENTS")
  )
)

pat_new_and_total_treatments = pat_new_and_total_treatments.withColumn("PULL_DT", F.current_date())

pat_new_and_total_treatments.filter(col("PATIENT_ID") == 29370266).display()

PATIENT_ID,SVC_DT_M,TOTAL_TREATMENTS,TOTAL_NEW_TREATMENTS,PULL_DT
29370266,2023-01-01,4,0,2026-10-19
29370266,2023-02-01,5,1,2026-10-19
29370266,2023-03-01,5,0,2026-10-19
29370266,2023-04-01,5,0,2026-10-19
29370266,2023-05-01,5,0,2026-10-19
29370266,2023-06-01,5,0,2026-10-19
29370266,2023-07-01,5,0,2026-10-19
29370266,2023-08-01,6,1,2026-10-19
29370266,2023-09-01,6,0,2026-10-19
29370266,2023-10-01,6,0,2026-10-19


In [0]:
pat_new_ckd_stage =  (
  df_spine_pat_journey_with_flag2
  # DATE is already at cohort-month grain from upstream notebooks
  .withColumn("SVC_DT_M", col("DATE"))
  .withColumn("CKD_STAGE", col("GFR_DX_STAGE"))
  .groupBy("PATIENT_ID", "SVC_DT_M")
  .agg(F.max("CKD_STAGE").alias("CKD_STAGE"))
  .withColumn("CKD_STAGE_LAST", F.lag("CKD_STAGE", 1).over(Window.partitionBy("PATIENT_ID").orderBy("SVC_DT_M")))
  .withColumn("NEW_CKD_STAGE_FLAG", F.when(col("CKD_STAGE") == col("CKD_STAGE_LAST"), 0).otherwise(1))
  .withColumn("NEW_CKD_STAGE", F.when(col("NEW_CKD_STAGE_FLAG") == 1, col("CKD_STAGE")).otherwise(None))
  .select("PATIENT_ID", "SVC_DT_M", "NEW_CKD_STAGE")
)

pat_new_ckd_stage = pat_new_ckd_stage.withColumn("PULL_DT", F.current_date())

pat_new_ckd_stage.filter(col("PATIENT_ID") == 29370266).display()

PATIENT_ID,SVC_DT_M,NEW_CKD_STAGE,PULL_DT
29370266,2023-01-01,null,2026-10-19
29370266,2023-02-01,null,2026-10-19
29370266,2023-03-01,null,2026-10-19
29370266,2023-04-01,null,2026-10-19
29370266,2023-05-01,null,2026-10-19
29370266,2023-06-01,null,2026-10-19
29370266,2023-07-01,null,2026-10-19
29370266,2023-08-01,null,2026-10-19
29370266,2023-09-01,null,2026-10-19
29370266,2023-10-01,null,2026-10-19


In [0]:
pat_gfr_dx_stage_and_treatment_category = (
  df_spine_pat_journey_with_flag2
  # DATE is already at cohort-month grain from upstream notebooks
  .withColumn("SVC_DT_M", col("DATE"))
  .withColumn("TREATMENT_CATEGORY", F.explode_outer(col("TREATMENT_CATEGORY")))
  .withColumn("CATEGORY", F.explode_outer(col("CATEGORY")))
  .groupBy("PATIENT_ID", "SVC_DT_M")
  .agg(F.max("GFR_DX_STAGE").alias("GFR_DX_STAGE"),
      F.max("GFR_UACR_DX_STAGE").alias("GFR_UACR_DX_STAGE"),
      F.max("UACR_STAGE").alias("UACR_STAGE"),
      F.collect_set("TREATMENT_CATEGORY").alias("CATEGORY"),
      F.collect_set("CATEGORY").alias("BRAND"))
)

pat_gfr_dx_stage_and_treatment_category = pat_gfr_dx_stage_and_treatment_category.withColumn("PULL_DT", F.current_date())

pat_gfr_dx_stage_and_treatment_category.filter(col("PATIENT_ID") == 29370266).display()

PATIENT_ID,SVC_DT_M,GFR_DX_STAGE,GFR_UACR_DX_STAGE,UACR_STAGE,CATEGORY,BRAND,PULL_DT
29370266,2025-09-01,null,null,null,"List(ACE/ ARB, Combo drug, Calcium channel / Beta blocker, MRA)","List(ACE/ ARB, Combo drug, Calcium channel / Beta blocker, MRA)",2026-10-19
29370266,2025-05-01,null,null,null,"List(Diuretic_NON_MRA, Calcium channel / Beta blocker, MRA, ACE/ ARB, Combo drug)","List(MRA, Combo drug, ACE/ ARB, Diuretic_NON_MRA, Calcium channel / Beta blocker)",2026-10-19
29370266,2023-02-01,null,null,null,"List(metformin , MRA, ACE/ ARB, Calcium channel / Beta blocker, Combo drug)","List(ACE/ ARB, Combo drug, metformin , Calcium channel / Beta blocker, MRA)",2026-10-19
29370266,2023-11-01,null,null,null,"List(metformin , Combo drug, Calcium channel / Beta blocker, MRA, SGLT2, ACE/ ARB)","List(metformin , Combo drug, Calcium channel / Beta blocker, ACE/ ARB, MRA, JARDIANCE)",2026-10-19
29370266,2024-09-01,null,null,null,"List(metformin , ACE/ ARB, MRA, Calcium channel / Beta blocker, Diuretic_NON_MRA, Combo drug)","List(Diuretic_NON_MRA, MRA, Calcium channel / Beta blocker, metformin , Combo drug, ACE/ ARB)",2026-10-19
29370266,2025-07-01,null,null,null,"List(ACE/ ARB, Combo drug, Calcium channel / Beta blocker, Diuretic_NON_MRA, MRA)","List(Diuretic_NON_MRA, MRA, Calcium channel / Beta blocker, ACE/ ARB, Combo drug)",2026-10-19
29370266,2023-04-01,null,null,null,"List(ACE/ ARB, MRA, Combo drug, metformin , Calcium channel / Beta blocker)","List(metformin , ACE/ ARB, Calcium channel / Beta blocker, Combo drug, MRA)",2026-10-19
29370266,2024-11-01,null,null,null,"List(Diuretic_NON_MRA, Combo drug, MRA, Calcium channel / Beta blocker, ACE/ ARB, metformin )","List(MRA, Calcium channel / Beta blocker, Diuretic_NON_MRA, ACE/ ARB, metformin , Combo drug)",2026-10-19
29370266,2024-02-01,null,null,null,"List(ACE/ ARB, metformin , Diuretic_NON_MRA, MRA, Calcium channel / Beta blocker, Combo drug)","List(Combo drug, MRA, ACE/ ARB, Calcium channel / Beta blocker, Diuretic_NON_MRA, metformin )",2026-10-19
29370266,2025-11-01,null,null,null,"List(Combo drug, MRA, ACE/ ARB, Calcium channel / Beta blocker)","List(Calcium channel / Beta blocker, ACE/ ARB, MRA, Combo drug)",2026-10-19


In [0]:
pat_uacr_stage_and_treatment_category = (
  df_spine_pat_journey_with_flag2
  # DATE is already at cohort-month grain from upstream notebooks
  .withColumn("SVC_DT_M", col("DATE"))
  .withColumn("TREATMENT_CATEGORY", F.explode_outer(col("TREATMENT_CATEGORY")))
  .withColumn("CATEGORY", F.explode_outer(col("CATEGORY")))
  .groupBy("PATIENT_ID", "SVC_DT_M")
  .agg(F.max("GFR_DX_STAGE").alias("GFR_DX_STAGE"),
      F.max("GFR_UACR_DX_STAGE").alias("GFR_UACR_DX_STAGE"),
      F.max("UACR_STAGE").alias("UACR_STAGE"),
      F.collect_set("TREATMENT_CATEGORY").alias("CATEGORY"),
      F.collect_set("CATEGORY").alias("BRAND"))
  .withColumn("CATEGORY", F.explode_outer(col("CATEGORY")))
  .withColumn("BRAND", F.explode_outer(col("BRAND")))
  .withColumn('CATEGORY', F.regexp_replace('CATEGORY', r'[\s]*$', ''))
)

pat_uacr_stage_and_treatment_category = pat_uacr_stage_and_treatment_category.withColumn("PULL_DT", F.current_date())

pat_uacr_stage_and_treatment_category.filter(col("PATIENT_ID") == 29370266).display()

PATIENT_ID,SVC_DT_M,GFR_DX_STAGE,GFR_UACR_DX_STAGE,UACR_STAGE,CATEGORY,BRAND,PULL_DT
29370266,2025-09-01,null,null,null,ACE/ ARB,ACE/ ARB,2026-10-19
29370266,2025-09-01,null,null,null,ACE/ ARB,Combo drug,2026-10-19
29370266,2025-09-01,null,null,null,ACE/ ARB,Calcium channel / Beta blocker,2026-10-19
29370266,2025-09-01,null,null,null,ACE/ ARB,MRA,2026-10-19
29370266,2025-09-01,null,null,null,Combo drug,ACE/ ARB,2026-10-19
29370266,2025-09-01,null,null,null,Combo drug,Combo drug,2026-10-19
29370266,2025-09-01,null,null,null,Combo drug,Calcium channel / Beta blocker,2026-10-19
29370266,2025-09-01,null,null,null,Combo drug,MRA,2026-10-19
29370266,2025-09-01,null,null,null,Calcium channel / Beta blocker,ACE/ ARB,2026-10-19
29370266,2025-09-01,null,null,null,Calcium channel / Beta blocker,Combo drug,2026-10-19


In [0]:
pat_uacr_stage_and_treatment_category.select('UACR_STAGE').distinct().display()

UACR_STAGE
2
3
1
null


In [0]:
pat_sglt2_flag = (
  df_spine_pat_journey_with_flag2
  .withColumn("TREATMENT_CATEGORY", F.explode_outer(col("TREATMENT_CATEGORY")))
  .filter("TREATMENT_CATEGORY = 'SGLT2'")
  .withColumn("SGLT2_FLAG", F.lit(1))
  .select("PATIENT_ID", "SGLT2_FLAG").distinct()
)



pat_sglt2_filter = (
  df_spine_pat_journey_with_flag2
  .withColumn("STAGE_ELIGIBLE", 
              F.when((col("GFR_UACR_DX_STAGE").isNull())|(col("GFR_UACR_DX_STAGE") == 0), -1).otherwise(
                F.when((col("GFR_UACR_DX_STAGE") == 1)|(col("GFR_UACR_DX_STAGE") == 5), 0).otherwise(1)))
  .join(pat_sglt2_flag, how = "left", on = "PATIENT_ID")
  .na.fill(value = 0, subset = "SGLT2_FLAG")
  .withColumn("ELIGIBLE_PATIENT_FLAG", 
              F.when((col("STAGE_ELIGIBLE") == 0), 0).otherwise(
                #F.when((col("GFR_UACR_DX_STAGE") == 1) & (col("DIALYSIS_FLAG") == 1), 0).otherwise(
                F.when((col("DIALYSIS_FLAG") == 1), 0).otherwise(
                F.when((col("GFR_UACR_DX_STAGE") == -1) & (col("SGLT2_FLAG") == 0), 0).otherwise(1))))
)



pat_sglt2_filter_m = (
  pat_sglt2_filter
  # DATE is already at cohort-month grain from upstream notebooks
  .withColumn("SVC_DT_M", col("DATE"))
  .groupBy("PATIENT_ID", "SVC_DT_M")
  .agg(F.max("ELIGIBLE_PATIENT_FLAG").alias("ELIGIBLE_PATIENT_FLAG"),
      F.max("GFR_UACR_DX_STAGE").alias("GFR_UACR_DX_STAGE"),
      F.max("GFR_DX_STAGE").alias("GFR_DX_STAGE"),
      F.max("UACR_STAGE").alias("UACR_STAGE"))
  .withColumn("GFR_UACR_DX_STAGE", 
            F.when((col("GFR_UACR_DX_STAGE") == 2), "STAGE_2").otherwise(
            F.when((col("GFR_UACR_DX_STAGE") == 3), "STAGE_3").otherwise(
            F.when((col("GFR_UACR_DX_STAGE") == 4), "STAGE_4").otherwise("STAGE_UNKNOWN"))))
  .withColumn("GFR_DX_STAGE", 
            F.when((col("GFR_DX_STAGE") == 2), "STAGE_2").otherwise(
            F.when((col("GFR_DX_STAGE") == 3), "STAGE_3").otherwise(
            F.when((col("GFR_DX_STAGE") == 4), "STAGE_4").otherwise("STAGE_UNKNOWN"))))
  .withColumn("UACR_STAGE", 
            F.when((col("UACR_STAGE") == 2), "STAGE_2").otherwise(
            F.when((col("UACR_STAGE") == 3), "STAGE_3").otherwise("STAGE_UNKNOWN")))
)

pat_sglt2_filter_m = pat_sglt2_filter_m.withColumn("PULL_DT", F.current_date())

pat_sglt2_filter_m.filter(col("PATIENT_ID") == 29370266).display()

PATIENT_ID,SVC_DT_M,ELIGIBLE_PATIENT_FLAG,GFR_UACR_DX_STAGE,GFR_DX_STAGE,UACR_STAGE,PULL_DT
29370266,2025-07-01,1,STAGE_UNKNOWN,STAGE_UNKNOWN,STAGE_UNKNOWN,2026-10-19
29370266,2025-03-01,1,STAGE_UNKNOWN,STAGE_UNKNOWN,STAGE_UNKNOWN,2026-10-19
29370266,2024-01-01,1,STAGE_UNKNOWN,STAGE_UNKNOWN,STAGE_UNKNOWN,2026-10-19
29370266,2025-12-01,1,STAGE_UNKNOWN,STAGE_UNKNOWN,STAGE_UNKNOWN,2026-10-19
29370266,2024-12-01,1,STAGE_UNKNOWN,STAGE_UNKNOWN,STAGE_UNKNOWN,2026-10-19
29370266,2024-11-01,1,STAGE_UNKNOWN,STAGE_UNKNOWN,STAGE_UNKNOWN,2026-10-19
29370266,2025-02-01,1,STAGE_UNKNOWN,STAGE_UNKNOWN,STAGE_UNKNOWN,2026-10-19
29370266,2025-11-01,1,STAGE_UNKNOWN,STAGE_UNKNOWN,STAGE_UNKNOWN,2026-10-19
29370266,2025-09-01,1,STAGE_UNKNOWN,STAGE_UNKNOWN,STAGE_UNKNOWN,2026-10-19
29370266,2026-03-01,1,STAGE_UNKNOWN,STAGE_UNKNOWN,STAGE_UNKNOWN,2026-10-19


In [0]:
# VALIDATION: write to separate paths
pat_new_and_total_treatments.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_pat_new_and_total_treatments_validation")

In [0]:
pat_new_ckd_stage.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_pat_new_ckd_stage_validation")

In [0]:
pat_gfr_dx_stage_and_treatment_category.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_pat_gfr_dx_stage_validation")

In [0]:
pat_uacr_stage_and_treatment_category.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_pat_uacr_stage_validation")

In [0]:
pat_sglt2_filter_m.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_pat_sglt2_filter_m_validation")

In [0]:
count_check = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_pat_sglt2_filter_m_validation")
count_check.count()

142742